In [0]:

%sql
select * from u_catalog.bronze.silver_obt

In [0]:
# TESTING CHANGES FOR SCD TYPE 2
df = spark.read.table("u_catalog.bronze.silver_obt")
df = df.select("pickup_city_id", "pickup_city", "region",  "state")
df = df.dropDuplicates(subset=['pickup_location_id'])
display(df)

In [0]:
df = spark.read.table("u_catalog.bronze.silver_obt")
df = df.select("vehicle_id", "vehicle_make_id", "vehicle_type_id", "vehicle_model", "vehicle_color", "license_plate", "vehicle_make", "vehicle_type", "description", "per_mile",)
df = df.dropDuplicates(subset=['vehicle_id'])
display(df)

In [0]:

df = spark.readStream.table("u_catalog.bronze.silver_obt")
df = df.select("distance_miles", "duration_minutes", "base_fare", "distance_fare", "time_fare", "surge_multiplier", "tip_amount","total_fare","rating", "base_rate", "per_mile", "per_minute")
df = df.dropDuplicates(subset=['ride_id'])

In [0]:
#bulk rides PLAY SMART

df = spark.sql("select * from u_catalog.bronze.bulk_rides")
df.schema



## Testing Gold Layer
Filter to avoid duplicates when joining with fact table

In [0]:
%sql 
select fact.ride_id, fact.base_fare from u_catalog.bronze.fact_rides as fact
left join u_catalog.bronze.dim_location dim
on fact.pickup_city_id= dim.pickup_city_id
and dim.`__END_AT` is null --- FILTER  

In [0]:
%sql
select * from u_catalog.bronze.dim_location --- TESTING CHANGES FOR SCD TYPE 2

In [0]:
%sql
-- Check current state of dim_location
SELECT * FROM u_catalog.bronze.dim_location
ORDER BY pickup_city_id


In [0]:
%sql
select 
    pickup_city_id,
    pickup_city,
    state,
    __START_AT,
    __END_AT
from u_catalog.bronze.dim_location
--where pickup_city_id = 1
order by __START_AT

In [0]:
%sql
select * from u_catalog.bronze.map_cities

In [0]:
%sql
select distinct pickup_city_id, pickup_city, state, city_updated_at
from u_catalog.bronze.silver_obt
order by pickup_city_id

## JINJA:

In [0]:
# Renders AGAIN and runs SQL
template = Template(jinja_str)
rendered_template = template.render(jinja_config=jinja_config)
display(spark.sql(rendered_template))
